In [1]:
import torch
print("GPU disponible :", torch.cuda.is_available())
print("Nom du GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Aucun GPU détecté")

GPU disponible : True
Nom du GPU : NVIDIA GeForce RTX 4060 Ti


In [5]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, SubsetRandomSampler,Subset
import matplotlib.pyplot as plt
import cv2
import torchvision.datasets as datasets
import torchvision.models as models
import torchvision.transforms.v2 as transforms
from tqdm import tqdm
from torchmetrics import Accuracy
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger, TensorBoardLogger , CSVLogger
from torchmetrics.classification import MulticlassConfusionMatrix
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from PIL import Image
import pandas as pd
import torch.nn.functional as F

In [2]:
import wandb
wandb.login(key="7d23279d258aea69ee52f79824db6fe34e7a3243")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\Sacha\_netrc
wandb: Currently logged in as: 231418 (231418-universit-de-mons) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
dataset_path = "C:/Users/Sacha/Desktop/Mini-Projet/Classification/railway-construction-big"  #@param ["dataset/railway-construction-50/","dataset/railway-construction-100/","dataset/railway-construction-big/"] {type:"string"}
batch_size = 16 #@param [8,16,32,64,128,256] {type:"raw"}
train_split = 0.85 #@param {type:"slider", min:0.5, max:0.9, step:0.05}
val_split = 0.1 #@param {type:"slider", min:0.1, max:0.5, step:0.05}
epochs = 30 #@param [1,5, 10,20,50,100,200] {type:"raw"}
learning_rate = 1.610731743873425e-05  #@param [0.1, 0.01,0.02,0.05,0.001,0.002,0.005] {type:"raw"}
img_size = 224
num_classes = 4
LOG_DIR = "logs/"

In [6]:
class Model(pl.LightningModule):
    def __init__(self, optimizer=torch.optim.AdamW, num_classes=4, learning_rate=3e-4):
        super().__init__()
        self.model = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
        self.model.heads.head = nn.Linear(self.model.heads.head.in_features,num_classes)
        self.criterion = nn.CrossEntropyLoss()
        self.lr = learning_rate
        self.num_classes = num_classes
        self.test_acc = Accuracy(task="multiclass", num_classes=self.num_classes)
        self.val_acc = Accuracy(task="multiclass", num_classes=self.num_classes)
        self.train_acc = Accuracy(task="multiclass", num_classes=self.num_classes)
        self.optimizer = optimizer
        self.confusion_matrix = MulticlassConfusionMatrix(num_classes=self.num_classes)

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        inputs, labels = batch
        outputs = self(inputs)
        loss = self.criterion(outputs, labels)
        _, predicted = torch.max(outputs, 1)  # Extraire les classes prédiction
        acc = self.val_acc(predicted, labels)
        self.log_dict({'train_loss':loss,"train_acc":acc}, on_step=True,prog_bar=True,logger=True, on_epoch=True)
        return loss

    def on_train_epoch_end(self):
        self.train_acc.reset()
        self.confusion_matrix.reset()
    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        val_loss = F.cross_entropy(y_hat, y)
        val_acc = self.val_acc(y_hat, y)
        self.log_dict({'val_loss':val_loss,"val_acc":val_acc}, on_step=False, on_epoch=True)
        self.confusion_matrix.update(y_hat.argmax(dim=1), y)
    def on_validation_epoch_end(self):
        self.val_acc.reset()
    def test_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        test_loss = F.cross_entropy(y_hat, y)
        test_acc = self.test_acc(y_hat, y)
        self.log_dict({'test_loss':test_loss,"test_acc":test_acc}, on_step=False, on_epoch=True)
        self.confusion_matrix.update(y_hat.argmax(dim=1), y)
        return test_loss

    def on_test_end(self):
        self.test_acc.reset()

    def configure_optimizers(self):
        return self.optimizer(self.model.parameters(), lr=self.lr,weight_decay=1e-4)

In [7]:
def create_data_loaders(dataset_path, batch_size, train_split, val_split, img_size):
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(img_size),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor()
    ])

    eval_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor()
    ])

    # Load the full dataset (without transforms initially)
    full_dataset = datasets.ImageFolder(dataset_path)

    dataset_size = len(full_dataset)
    indices = list(range(dataset_size))
    np.random.shuffle(indices)

    # Split indices
    train_end = int(train_split * dataset_size)
    val_end = train_end + int(val_split * dataset_size)

    train_indices = indices[:train_end]
    val_indices = indices[train_end:val_end]
    test_indices = indices[val_end:]

    # Create separate dataset subsets with corresponding transformations
    train_dataset = Subset(datasets.ImageFolder(dataset_path, transform=train_transform), train_indices)
    val_dataset = Subset(datasets.ImageFolder(dataset_path, transform=eval_transform), val_indices)
    test_dataset = Subset(datasets.ImageFolder(dataset_path, transform=eval_transform), test_indices)

    # Create DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

In [8]:
train_loader, val_loader, test_loader = create_data_loaders(dataset_path, batch_size, train_split, val_split, img_size)

C:\Users\Sacha\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\torchvision\transforms\v2\_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


In [ ]:
wandb_logger = WandbLogger(
    project="MINI-Projet",
    name="VitB16",
    log_model=True,
)
# Initialize model
optimizer = torch.optim.SGD
model = Model(learning_rate=learning_rate)

# Initialize CsvLogger
csv_logger = CSVLogger(LOG_DIR, name="VitB16", version='')

checkpoint_callback = ModelCheckpoint(
monitor='val_loss',
dirpath='checkpoints',
filename='best-checkpoint',
save_top_k=1,
mode='min')

early_stop_callback = EarlyStopping(
monitor='val_loss'
,
patience=10,
mode='min')

# Initialize Trainer
trainer = pl.Trainer(
    max_epochs=epochs,
    accelerator="auto",
    logger=[wandb_logger, csv_logger],
    callbacks=[checkpoint_callback, early_stop_callback],
)

# Train the model
trainer.fit(model, train_loader, val_loader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 4060 Ti') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name             | Type                      | Params | Mode 
-----------------------------------------------------------------------
0 | model            | VisionTransformer         | 85.8 M | train
1 | criterion        | CrossEntropyLoss          | 0      | train
2 | test_acc         | MulticlassAccuracy        | 0      | train
3 | val_acc          | MulticlassAccuracy        | 0      | train
4 | train_acc        | MulticlassAccuracy        | 0      | train
5 | confusion_matrix | MulticlassConfusionMatrix | 0      | train
-----------------------------------------------------------------------
85.8 M    Trainable params
0         Non-trainable params
85.8 M    Total params
343.207   Total estimated model params size (MB)
157       Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

C:\Users\Sacha\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


C:\Users\Sacha\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Epoch 17:  11%|█         | 12/110 [00:05<00:41,  2.35it/s, v_num=o63_, train_loss_step=0.00172, train_acc_step=1.000, train_loss_epoch=0.0597, train_acc_epoch=0.978] 

In [ ]:
trainer.test(model, test_loader)
wandb.finish()

In [ ]:
jit_model = model.to_torchscript()
torch.jit.save(jit_model, 'model_jitb.pth')

In [ ]:
test_folder_path  = 'C:/Users/Sacha/Desktop/Mini-Projet/Classification/test'

In [ ]:
test_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor()
    ])

In [ ]:
def predict_folder(model, folder_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    model.eval()


    predictions = []
    filenames = []

    for img_path in os.listdir(folder_path):
        with torch.no_grad():
            # Load and preprocess image
            image = Image.open(folder_path+'/'+img_path).convert('RGB')
            image = test_transform(image).unsqueeze(0).to(device)

            # Get prediction
            output = model(image)
            _, predicted = output.max(1)

            predictions.append(predicted.item())
            filenames.append(img_path)

    # Create submission dataframe
    submission_df = pd.DataFrame({
        'ID': filenames,
        'Label': predictions
    })

    # Save to CSV
    submission_df.to_csv('submissionbig.csv', index=False)
    return submission_df


In [ ]:
submission = predict_folder(jit_model, test_folder_path)

In [ ]:
import torch
from torchvision.models import resnet50, ResNet50_Weights
from pytorch_bench import benchmark

# Load model and example input
model = resnet50(weights=ResNet50_Weights.DEFAULT)
example_input = torch.randn(1, 3, 224, 224)

# Run benchmark
results = benchmark(model, example_input)